# Transferibilidad a Colombia — 02: Curvas de Rendimiento y Analisis Estadistico

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/transferibilidad_02_curvas_rendimiento.ipynb)

**Analisis incluidos:**
1. Curva de aprendizaje — cuantos datos necesita cada modelo
2. Curvas Precision-Recall y Average Precision (AP)
3. Test de McNemar — significancia estadistica entre modelos
4. Analisis por tamano de deslizamiento — donde falla el modelo
5. Ensamble RF+CNN — mejora por combinacion de modelos

> Cambia `DRIVE_PATH` antes de correr.

## 0. Configuracion

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado')

In [ ]:
from pathlib import Path
import sys
DRIVE_PATH='/content/drive/MyDrive/Landslide4Sense'
ROOT=Path(DRIVE_PATH); IMG_DIR=ROOT/'TrainData'/'img'; MASK_DIR=ROOT/'TrainData'/'mask'
OUT_DIR=ROOT/'results'/'transferibilidad_02'; OUT_DIR.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(ROOT))
print(f'Imagenes: {len(list(IMG_DIR.glob("*.h5")))}  |  Salida: {OUT_DIR}')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import h5py, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (f1_score,precision_recall_curve,average_precision_score,
                             roc_auc_score,confusion_matrix)
from scipy.stats import chi2
from tqdm.notebook import tqdm
%matplotlib inline
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})

CHANNEL_MEAN=[0.1245,0.1438,0.1312,0.2891,0.3015,0.2134,0.1789,0.0823,0.0641,
              0.4521,0.2189,0.3102,0.3478,0.3812]
CHANNEL_STD=[0.0512,0.0621,0.0589,0.0934,0.0978,0.0734,0.0612,0.0341,0.0289,
             0.2134,0.1456,0.0812,0.0867,0.0923]
N_TOTAL=1500; SEED=42
print('Imports OK')

## 1. Carga del dataset completo

In [ ]:
def normalize(patch):
    m=np.array(CHANNEL_MEAN,dtype=np.float32).reshape(1,1,-1)
    s=np.array(CHANNEL_STD, dtype=np.float32).reshape(1,1,-1)
    return (patch-m)/(s+1e-8)

def extract_features(patch):
    eps=1e-8
    nir,rojo,verde,azul=patch[:,:,3],patch[:,:,2],patch[:,:,1],patch[:,:,0]
    vv,vh=patch[:,:,7],patch[:,:,8]
    ndvi=(nir-rojo)/(nir+rojo+eps); ndwi=(verde-nir)/(verde+nir+eps)
    sar_cr=vh/(vv+eps); evi=2.5*(nir-rojo)/(nir+6*rojo-7.5*azul+1+eps)
    return np.concatenate([patch.mean(axis=(0,1)),patch.std(axis=(0,1)),
                           [ndvi.mean(),ndwi.mean(),sar_cr.mean(),evi.mean()]]).astype(np.float32)

def load_dataset(n=N_TOTAL,seed=SEED):
    img_files=sorted(IMG_DIR.glob('*.h5'))
    rng=np.random.default_rng(seed)
    chosen=sorted(rng.choice(len(img_files),size=min(n,len(img_files)),replace=False))
    X_list,y_list,coverage_list=[],[],[]
    for i in tqdm(chosen,desc='Cargando'):
        img_f=img_files[i]; mask_f=MASK_DIR/img_f.name.replace('image_','mask_')
        if not mask_f.exists(): continue
        with h5py.File(img_f,'r') as f: patch=f['img' if 'img' in f else list(f.keys())[0]][()].astype(np.float32)
        with h5py.File(mask_f,'r') as f: mask=f['mask' if 'mask' in f else list(f.keys())[0]][()]
        X_list.append(extract_features(normalize(patch)))
        y_list.append(int(mask.max()>0))
        coverage_list.append(float(mask.mean()))  # fraccion de pixeles positivos
    X,y=np.stack(X_list),np.array(y_list)
    print(f'Dataset: {X.shape[0]} | positivos: {y.sum()} ({100*y.mean():.1f}%)')
    return X,y,np.array(coverage_list)

X,y,coverage=load_dataset()
print(f'Shape: {X.shape}')

## 2. Curva de aprendizaje

F1 en funcion del numero de muestras de entrenamiento.
**Hipotesis central del proyecto:** la brecha RF vs U-Net se explica por volumen de datos.

In [ ]:
checkpoints=[50,100,150,200,300,400,500,700,1000,1200,len(X)]
checkpoints=[c for c in checkpoints if c<=len(X)]

X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)

print(f'Train: {len(X_tr)}  Test: {len(X_te)}')
print('Calculando curva de aprendizaje...')

lc_f1,lc_auc,lc_std=[],[],[]
for n in tqdm(checkpoints,desc='Checkpoints'):
    idx=np.random.default_rng(SEED).choice(len(X_tr),size=min(n,len(X_tr)),replace=False)
    Xn,yn=X_tr[idx],y_tr[idx]
    if yn.sum()==0 or yn.sum()==len(yn): lc_f1.append(0); lc_auc.append(0.5); lc_std.append(0); continue
    rf=RandomForestClassifier(n_estimators=150,class_weight='balanced',
                              min_samples_leaf=2,n_jobs=-1,random_state=SEED)
    rf.fit(Xn,yn)
    prob=rf.predict_proba(X_te)[:,1]
    lc_f1.append(f1_score(y_te,rf.predict(X_te),zero_division=0))
    lc_auc.append(roc_auc_score(y_te,prob))
    # Varianza entre 5 subsamples
    sub=[]
    for s in range(5):
        si=np.random.default_rng(s).choice(len(X_tr),size=min(n,len(X_tr)),replace=False)
        rf2=RandomForestClassifier(n_estimators=100,class_weight='balanced',n_jobs=-1,random_state=s)
        rf2.fit(X_tr[si],y_tr[si])
        sub.append(f1_score(y_te,rf2.predict(X_te),zero_division=0))
    lc_std.append(np.std(sub))

print('Curva calculada')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))

ax=axes[0]
ax.plot(checkpoints,lc_f1,'o-',color='#2E5FA3',lw=2,ms=6,label='RF F1-Score')
ax.fill_between(checkpoints,
                [f-s for f,s in zip(lc_f1,lc_std)],
                [f+s for f,s in zip(lc_f1,lc_std)],
                alpha=0.15,color='#2E5FA3',label='IC (+/- 1 std)')
# Referencias de literatura
ax.axhline(0.8700,color='#16A34A',lw=1.5,ls='--',label='SOTA L4S (3799 muestras) F1=0.870')
ax.axhline(0.5992,color='#D97706',lw=1.5,ls='--',label='U-Net baseline L4S F1=0.599')
ax.axvline(1900, color='gray',lw=1,ls=':',label='~1900 muestras (2-fold CV)')
ax.set_xlabel('Muestras de entrenamiento',fontsize=11)
ax.set_ylabel('F1-Score',fontsize=11)
ax.set_title('Curva de aprendizaje — RF\n(hipotesis: mas datos = modelo mas competitivo)',
             fontsize=11,fontweight='bold')
ax.legend(fontsize=8); ax.set_ylim(0,1)
ax.grid(axis='y',linestyle='--',alpha=0.4)

ax=axes[1]
ax.plot(checkpoints,lc_auc,'s-',color='#9333EA',lw=2,ms=6,label='RF AUC-ROC')
ax.set_xlabel('Muestras de entrenamiento',fontsize=11)
ax.set_ylabel('AUC-ROC',fontsize=11)
ax.set_title('Curva de aprendizaje — AUC-ROC',fontsize=11,fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0.5,1)
ax.grid(axis='y',linestyle='--',alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR/'curva_aprendizaje.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: curva_aprendizaje.png')

## 3. Curvas Precision-Recall y Average Precision (AP)

Mas informativas que ROC para datasets desbalanceados. El AP resume el area bajo la curva PR.

In [ ]:
# Entrenar RF final con todos los datos de entrenamiento
rf_final=RandomForestClassifier(n_estimators=200,class_weight='balanced',
                                min_samples_leaf=2,n_jobs=-1,random_state=SEED)
rf_final.fit(X_tr,y_tr)
prob_te=rf_final.predict_proba(X_te)[:,1]

prec,rec,thresh=precision_recall_curve(y_te,prob_te)
ap=average_precision_score(y_te,prob_te)

# F1 por umbral
f1_thresh=[2*p*r/(p+r+1e-8) for p,r in zip(prec[:-1],rec[:-1])]
best_idx=np.argmax(f1_thresh)
best_thresh=thresh[best_idx]; best_f1=f1_thresh[best_idx]

fig,axes=plt.subplots(1,2,figsize=(13,5))

ax=axes[0]
ax.step(rec,prec,where='post',color='#2E5FA3',lw=2,label=f'RF (AP={ap:.4f})')
ax.scatter(rec[best_idx],prec[best_idx],s=120,color='#EF4444',zorder=5,
           label=f'Umbral optimo p={best_thresh:.2f} (F1={best_f1:.4f})')
ax.set_xlabel('Recall (Sensibilidad)',fontsize=11)
ax.set_ylabel('Precision (VPP)',fontsize=11)
ax.set_title(f'Curva Precision-Recall\nAverage Precision = {ap:.4f}',
             fontsize=11,fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.grid(linestyle='--',alpha=0.3)

ax=axes[1]
ax.plot(thresh,f1_thresh,color='#16A34A',lw=2)
ax.axvline(best_thresh,color='#EF4444',lw=1.5,ls='--',
           label=f'Umbral optimo: {best_thresh:.3f}')
ax.axvline(0.5,color='gray',lw=1,ls=':',label='Umbral estandar: 0.5')
ax.set_xlabel('Umbral de decision p',fontsize=11)
ax.set_ylabel('F1-Score',fontsize=11)
ax.set_title('F1 vs Umbral de decision\n(el umbral optimo puede diferir de 0.5)',
             fontsize=11,fontweight='bold')
ax.legend(fontsize=9)
ax.grid(linestyle='--',alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR/'precision_recall.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'AP = {ap:.4f}  |  Umbral optimo = {best_thresh:.3f}  |  F1 optimo = {best_f1:.4f}')

## 4. Test de McNemar — Significancia estadistica entre modelos

Determina si las diferencias de F1 entre modelos son estadisticamente significativas
o podrian ser variabilidad aleatoria del fold.

In [ ]:
def mcnemar_test(y_true,pred_a,pred_b):
    'Test de McNemar entre dos vectores de predicciones binarias.'
    b=np.sum((pred_a==y_true)&(pred_b!=y_true))  # A correcto, B incorrecto
    c=np.sum((pred_a!=y_true)&(pred_b==y_true))  # A incorrecto, B correcto
    if b+c==0: return 1.0,1.0  # sin diferencia
    # Con correccion de continuidad de Edwards
    stat=(abs(b-c)-1)**2/(b+c)
    p_val=1-chi2.cdf(stat,df=1)
    return stat,p_val

# Generar predicciones de 3 variantes del RF para comparacion
print('Entrenando variantes...')
modelos_preds={}

# RF completo (linea base)
modelos_preds['RF 200 arboles']=rf_final.predict(X_te)

# RF con menos arboles
rf50=RandomForestClassifier(n_estimators=50,class_weight='balanced',n_jobs=-1,random_state=SEED)
rf50.fit(X_tr,y_tr); modelos_preds['RF 50 arboles']=rf50.predict(X_te)

# RF sin balanceo de clases
rf_nb=RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=SEED)
rf_nb.fit(X_tr,y_tr); modelos_preds['RF sin balanceo']=rf_nb.predict(X_te)

# RF con umbral optimo
modelos_preds['RF umbral optimo']=(prob_te>=best_thresh).astype(int)

# Tabla de McNemar
print('\n=== TEST DE McNEMAR (p < 0.05 = diferencia significativa) ===')
nombres=list(modelos_preds.keys()); ref='RF 200 arboles'
print(f'Referencia: {ref}  F1={f1_score(y_te,modelos_preds[ref],zero_division=0):.4f}')
print(f'{"Modelo":<25} {"F1":>8} {"stat":>8} {"p-valor":>10} {"Significativo?":>15}')
print('-'*70)
for nombre,pred in modelos_preds.items():
    if nombre==ref: continue
    f1=f1_score(y_te,pred,zero_division=0)
    stat,pval=mcnemar_test(y_te,modelos_preds[ref],pred)
    sig='SI ***' if pval<0.05 else 'no'
    print(f'{nombre:<25} {f1:>8.4f} {stat:>8.3f} {pval:>10.4f} {sig:>15}')

In [ ]:
# Figura: F1 con IC de McNemar
todos_nombres=list(modelos_preds.keys())
todos_f1=[f1_score(y_te,p,zero_division=0) for p in modelos_preds.values()]

fig,ax=plt.subplots(figsize=(9,4))
colors=['#2E5FA3']+['#16A34A' if f>todos_f1[0] else '#D97706' for f in todos_f1[1:]]
bars=ax.barh(range(len(todos_nombres)),todos_f1,color=colors,edgecolor='white')
ax.set_yticks(range(len(todos_nombres))); ax.set_yticklabels(todos_nombres,fontsize=10)
ax.invert_yaxis(); ax.set_xlabel('F1-Score',fontsize=11)
ax.axvline(todos_f1[0],color='navy',lw=1.5,ls='--',label='Referencia')
ax.set_title('Comparacion de variantes RF (test McNemar)',fontsize=11,fontweight='bold')
for i,v in enumerate(todos_f1):
    ax.text(v+0.002,i,f'{v:.4f}',va='center',fontsize=9)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR/'mcnemar_comparacion.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: mcnemar_comparacion.png')

## 5. Analisis por tamano de deslizamiento

Estratificar el F1 por cobertura del parche: pequeno (<10%), medio (10-50%), grande (>50%).
Los modelos clasicos suelen fallar mas en deslizamientos pequenos y fragmentados.

In [ ]:
# Definir estratos por coverage (fraccion de pixeles positivos)
estratos={
    'Negativos\n(0%)':    (coverage==0),
    'Pequeno\n(1-10%)':   (coverage>0)&(coverage<=0.10),
    'Medio\n(10-50%)':    (coverage>0.10)&(coverage<=0.50),
    'Grande\n(>50%)':     (coverage>0.50),
}

print('Distribucion por tamano:')
for nombre,mask in estratos.items():
    print(f'  {nombre.replace(chr(10)," "):<25}: {mask.sum()} parches ({100*mask.mean():.1f}%)')

# Evaluar RF en cada estrato
rf_all=RandomForestClassifier(n_estimators=200,class_weight='balanced',
                              min_samples_leaf=2,n_jobs=-1,random_state=SEED)
rf_all.fit(X,y)
y_pred_all=rf_all.predict(X)
prob_all=rf_all.predict_proba(X)[:,1]

print('\nF1 por estrato de tamano:')
estrato_f1,estrato_n=[],[]
for nombre,mask in estratos.items():
    if mask.sum()<5: estrato_f1.append(np.nan); estrato_n.append(mask.sum()); continue
    yv,yp=y[mask],y_pred_all[mask]
    f1=f1_score(yv,yp,zero_division=0)
    estrato_f1.append(f1); estrato_n.append(mask.sum())
    print(f'  {nombre.replace(chr(10)," "):<28}: F1={f1:.4f}  n={mask.sum()}')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))

nombres_e=list(estratos.keys())
ax=axes[0]
valid=[i for i,f in enumerate(estrato_f1) if not np.isnan(f)]
bars=ax.bar([nombres_e[i] for i in valid],
            [estrato_f1[i] for i in valid],
            color=['#6B7280','#F59E0B','#2E5FA3','#16A34A'],
            edgecolor='white',linewidth=0.8)
for bar,v in zip(bars,[estrato_f1[i] for i in valid]):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
            f'{v:.3f}',ha='center',va='bottom',fontsize=10,fontweight='bold')
ax.set_ylabel('F1-Score',fontsize=11)
ax.set_title('F1 por tamano de deslizamiento\n(pequenos suelen ser mas dificiles)',
             fontsize=11,fontweight='bold')
ax.set_ylim(0,1); ax.grid(axis='y',linestyle='--',alpha=0.4)

ax=axes[1]
ax.bar([nombres_e[i] for i in valid],
       [estrato_n[i] for i in valid],
       color=['#6B7280','#F59E0B','#2E5FA3','#16A34A'],
       edgecolor='white',linewidth=0.8)
ax.set_ylabel('Numero de parches',fontsize=11)
ax.set_title('Distribucion de parches por tamano',fontsize=11,fontweight='bold')
ax.grid(axis='y',linestyle='--',alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR/'analisis_tamano.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: analisis_tamano.png')

## 6. Ensamble de modelos

Promedio ponderado de probabilidades de 3 variantes de RF.
Los mejores resultados en L4S 2022 usaban ensambles (F1=0.870).

In [ ]:
# Entrenar 3 RF con diferentes semillas y parametros
print('Entrenando ensamble...')
rf1=RandomForestClassifier(n_estimators=200,max_features='sqrt',class_weight='balanced',
                           min_samples_leaf=2,n_jobs=-1,random_state=42).fit(X_tr,y_tr)
rf2=RandomForestClassifier(n_estimators=200,max_features='log2',class_weight='balanced',
                           min_samples_leaf=1,n_jobs=-1,random_state=7).fit(X_tr,y_tr)
rf3=RandomForestClassifier(n_estimators=300,max_features=0.5,class_weight='balanced',
                           min_samples_leaf=3,n_jobs=-1,random_state=13).fit(X_tr,y_tr)

p1=rf1.predict_proba(X_te)[:,1]
p2=rf2.predict_proba(X_te)[:,1]
p3=rf3.predict_proba(X_te)[:,1]

# Varios esquemas de ensamble
prom_simple=(p1+p2+p3)/3
# Pesos por F1 individual
f1s=[f1_score(y_te,(p>=0.5).astype(int),zero_division=0) for p in [p1,p2,p3]]
w=np.array(f1s)/sum(f1s)
prom_ponderado=w[0]*p1+w[1]*p2+w[2]*p3

resultados_ens={
    'RF1 solo':       f1_score(y_te,(p1>=0.5).astype(int),zero_division=0),
    'RF2 solo':       f1_score(y_te,(p2>=0.5).astype(int),zero_division=0),
    'RF3 solo':       f1_score(y_te,(p3>=0.5).astype(int),zero_division=0),
    'Ensamble simple':(f1_score(y_te,(prom_simple>=0.5).astype(int),zero_division=0)),
    'Ensamble ponderado':(f1_score(y_te,(prom_ponderado>=0.5).astype(int),zero_division=0)),
}
print('Resultados del ensamble:')
for k,v in resultados_ens.items():
    mejora=(v-resultados_ens['RF1 solo'])*100
    print(f'  {k:<22}: F1={v:.4f}  ({mejora:+.2f} pp)')

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
nombres_ens=list(resultados_ens.keys()); vals_ens=list(resultados_ens.values())
base_ens=vals_ens[0]
cols=['#6B7280','#6B7280','#6B7280','#2E5FA3','#16A34A']
bars=ax.barh(range(len(nombres_ens)),vals_ens,color=cols,edgecolor='white')
ax.set_yticks(range(len(nombres_ens))); ax.set_yticklabels(nombres_ens,fontsize=10)
ax.invert_yaxis(); ax.set_xlabel('F1-Score',fontsize=11)
ax.axvline(base_ens,color='gray',lw=1.2,ls='--')
ax.set_title('Impacto del ensamble de modelos RF',fontsize=11,fontweight='bold')
for i,v in enumerate(vals_ens):
    ax.text(v+0.001,i,f'{v:.4f}',va='center',fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR/'ensamble_modelos.png',dpi=150,bbox_inches='tight')
plt.show(); print('Guardado: ensamble_modelos.png')

## 7. Resumen

In [ ]:
print('='*65)
print('  RESUMEN — CURVAS DE RENDIMIENTO Y ANALISIS ESTADISTICO')
print('='*65)
print(f'\nCurva de aprendizaje:')
print(f'  F1 con 100 muestras : {lc_f1[1]:.4f}')
print(f'  F1 con 500 muestras : {lc_f1[[c for c in checkpoints].index(min(checkpoints,key=lambda x:abs(x-500)))]:.4f}')
print(f'  F1 maximo alcanzado : {max(lc_f1):.4f} ({checkpoints[lc_f1.index(max(lc_f1))]} muestras)')
print(f'\nPrecision-Recall:')
print(f'  AP (Average Precision) : {ap:.4f}')
print(f'  Umbral optimo          : {best_thresh:.3f}  (F1={best_f1:.4f})')
print(f'\nEnsamble (mejor)        : F1={max(resultados_ens.values()):.4f}')
print(f'\nFiguras guardadas en: {OUT_DIR}')